# Day 15 - Module 08: Custom Exceptions## Why Custom Exceptions?Built-in exceptions like `ValueError` and `TypeError` are generic.Custom exceptions give your code **meaningful, specific error types** that communicate exactly what went wrong.## Exception Hierarchy```BaseException  └── Exception        ├── ValueError        ├── TypeError        ├── RuntimeError        └── YourCustomException  ← Create your own here```## Creating Custom Exceptions```pythonclass MyError(Exception):    passclass DetailedError(Exception):    def __init__(self, message, code):        super().__init__(message)        self.code = code```

## Example 1: Banking Exception Hierarchy

In [ ]:
class BankingError(Exception):    """Base exception for all banking errors"""    passclass InsufficientFundsError(BankingError):    def __init__(self, required, available):        self.required  = required        self.available = available        super().__init__(            f"Insufficient funds: need Rs.{required}, have Rs.{available}"        )    @property    def shortage(self):        return self.required - self.availableclass InvalidPINError(BankingError):    def __init__(self, attempts_left):        self.attempts_left = attempts_left        super().__init__(f"Wrong PIN! Attempts remaining: {attempts_left}")class AccountLockedError(BankingError):    def __init__(self):        super().__init__("Account is locked. Please contact support.")class BankAccount:    MAX_PIN_ATTEMPTS = 3    def __init__(self, owner, pin, balance):        self.owner         = owner        self.__pin         = pin        self._balance      = balance        self._pin_attempts = 0        self._locked       = False    def withdraw(self, amount, pin):        if self._locked:            raise AccountLockedError()        if pin != self.__pin:            self._pin_attempts += 1            remaining = self.MAX_PIN_ATTEMPTS - self._pin_attempts            if remaining == 0:                self._locked = True                raise AccountLockedError()            raise InvalidPINError(remaining)        self._pin_attempts = 0  # reset on success        if amount > self._balance:            raise InsufficientFundsError(amount, self._balance)        self._balance -= amount        return self._balanceacc = BankAccount("Shivaji", 1234, 5000)# Test insufficient fundstry:    acc.withdraw(10000, 1234)except InsufficientFundsError as e:    print(f"Error: {e}")    print(f"Shortage: Rs.{e.shortage}")# Test wrong PINfor _ in range(4):    try:        acc.withdraw(100, 9999)    except InvalidPINError as e:        print(f"PIN Error: {e}")    except AccountLockedError as e:        print(f"Locked! {e}")

## Example 2: Form Validation Exceptions

In [ ]:
class ValidationError(Exception):    def __init__(self, field, message):        self.field   = field        self.message = message        super().__init__(f"[{field}] {message}")class RequiredFieldError(ValidationError):    def __init__(self, field):        super().__init__(field, "This field is required")class InvalidEmailError(ValidationError):    def __init__(self, email):        super().__init__("email", f"'{email}' is not a valid email address")class AgeRangeError(ValidationError):    def __init__(self, age, min_age=18, max_age=100):        super().__init__("age", f"Age {age} out of range [{min_age}-{max_age}]")def validate_registration(name, email, age):    errors = []    if not name.strip():        errors.append(RequiredFieldError("name"))    if "@" not in email or "." not in email:        errors.append(InvalidEmailError(email))    if not (18 <= age <= 100):        errors.append(AgeRangeError(age))    return errorstest_cases = [    ("", "notanemail", 15),    ("Shivaji", "shivaji@gmail.com", 22),]for name, email, age in test_cases:    errs = validate_registration(name, email, age)    if errs:        print(f"INVALID: {len(errs)} error(s):")        for e in errs:            print(f"  - {e}")    else:        print(f"VALID: Registration for {name} accepted")    print()

## Practice Questions### Q1: E-Commerce Order Exception SystemCreate a custom exception hierarchy:- `OrderError(Exception)` — base  - `OutOfStockError(OrderError)` — with `product_name` and `requested_qty`  - `InvalidCouponError(OrderError)` — with `coupon_code`  - `DeliveryUnavailableError(OrderError)` — with `pincode`Create class `OrderSystem` with method `place_order(product, qty, coupon, pincode)` that:- Raises `OutOfStockError` if qty > 5 (simulate stock limit)- Raises `InvalidCouponError` if coupon is not `"SAVE10"` or `"SAVE20"`- Raises `DeliveryUnavailableError` if pincode starts with `"00"`Test with valid and invalid scenarios.

In [ ]:
# Q1 Solution: E-Commerce Order Exceptionsclass OrderError(Exception):    passclass OutOfStockError(OrderError):    def __init__(self, product_name, requested_qty, available=5):        self.product_name  = product_name        self.requested_qty = requested_qty        self.available     = available        super().__init__(            f"'{product_name}' has only {available} units, requested {requested_qty}"        )class InvalidCouponError(OrderError):    def __init__(self, coupon_code):        self.coupon_code = coupon_code        super().__init__(f"Coupon '{coupon_code}' is invalid or expired")class DeliveryUnavailableError(OrderError):    def __init__(self, pincode):        self.pincode = pincode        super().__init__(f"Delivery not available at pincode {pincode}")class OrderSystem:    VALID_COUPONS = {"SAVE10", "SAVE20"}    STOCK_LIMIT   = 5    def place_order(self, product, qty, coupon, pincode):        if qty > self.STOCK_LIMIT:            raise OutOfStockError(product, qty, self.STOCK_LIMIT)        if coupon and coupon not in self.VALID_COUPONS:            raise InvalidCouponError(coupon)        if str(pincode).startswith("00"):            raise DeliveryUnavailableError(pincode)        print(f"Order placed! {qty}x {product} | Coupon: {coupon or 'None'} | PIN: {pincode}")orders = OrderSystem()test_orders = [    ("iPhone 15",  2, "SAVE10", "400001"),   # valid    ("iPad",       8, "SAVE10", "400001"),   # out of stock    ("MacBook",    1, "HACKME", "400001"),   # bad coupon    ("AirPods",    1, None,     "001234"),   # bad pincode]for product, qty, coupon, pincode in test_orders:    try:        orders.place_order(product, qty, coupon, pincode)    except OutOfStockError as e:        print(f"[OUT OF STOCK] {e}")    except InvalidCouponError as e:        print(f"[COUPON ERROR] {e}")    except DeliveryUnavailableError as e:        print(f"[DELIVERY ERR] {e}")

### Q2: File Processing ExceptionsCreate:- `FileProcessingError(Exception)` — base  - `EmptyFileError(FileProcessingError)` — file is empty  - `InvalidFormatError(FileProcessingError)` — with expected and actual format  - `FileSizeLimitError(FileProcessingError)` — with size and limitCreate function `process_file(filename, content, max_size_kb=10)`:- Raises `EmptyFileError` if content is empty- Raises `FileSizeLimitError` if `len(content) > max_size_kb * 1024`- Raises `InvalidFormatError` if filename doesn't end in `.csv` or `.txt`- Otherwise prints "File processed: X characters"Test all 4 scenarios.

In [ ]:
# Q2 Solution: File Processing Exceptionsclass FileProcessingError(Exception):    passclass EmptyFileError(FileProcessingError):    def __init__(self, filename):        super().__init__(f"File '{filename}' is empty")class InvalidFormatError(FileProcessingError):    def __init__(self, filename, expected, actual):        self.expected = expected        self.actual   = actual        super().__init__(            f"Invalid format for '{filename}': expected {expected}, got '{actual}'"        )class FileSizeLimitError(FileProcessingError):    def __init__(self, filename, size_kb, limit_kb):        self.size_kb  = size_kb        self.limit_kb = limit_kb        super().__init__(            f"'{filename}' is {size_kb:.1f} KB, exceeds limit of {limit_kb} KB"        )def process_file(filename, content, max_size_kb=10):    ext = filename.rsplit(".", 1)[-1] if "." in filename else ""    if not content:        raise EmptyFileError(filename)    size_kb = len(content) / 1024    if size_kb > max_size_kb:        raise FileSizeLimitError(filename, size_kb, max_size_kb)    if ext not in ("csv", "txt"):        raise InvalidFormatError(filename, "csv or txt", ext)    print(f"Processed '{filename}': {len(content)} characters OK")test_cases = [    ("data.csv",   "name,age\nAlice,30", 10),   # valid    ("empty.txt",  "",                    10),   # empty    ("report.pdf", "some content",        10),   # wrong format    ("big.csv",    "x" * 15000,            10),   # too big]for fname, content, max_kb in test_cases:    try:        process_file(fname, content, max_kb)    except EmptyFileError as e:        print(f"[EMPTY] {e}")    except FileSizeLimitError as e:        print(f"[TOO BIG] {e}")    except InvalidFormatError as e:        print(f"[FORMAT] {e}")

## Key Takeaways- Custom exceptions should inherit from `Exception` (or a custom base)- Add custom attributes (`self.field`, `self.code`) for extra context- Build hierarchies: `catch BankingError` to catch all banking-related exceptions- Use `super().__init__(message)` to set the exception message properly- Always catch the most **specific** exception first, then broad ones